Language detection inside the worker processes, so both language detection and tokenization run in parallel.

Key points:

The main process only streams and groups raw rows into chunks (no language filtering).

Each worker instantiates its own langid + tiktoken encoder, filters its chunk for English, tokenizes kept rows, saves a .npy chunk file, and returns a small verification sample (first up to 20 token ids + decoded text).

The main process logs each chunk's sample as soon as a worker returns it and then merges all .npy chunk files into one memmapped .npy.

In [1]:
#!/usr/bin/env python3
"""
Parallel parquet -> filter (in-worker) -> tokenize -> chunk-save pipeline (Option B)
with optional profiling harness.

- Streams parquet rows with pyarrow.dataset.
- Groups raw rows into text chunks up to TARGET_CHUNK_BYTES (approx bytes).
- Dispatches chunks in parallel to worker processes (joblib loky backend).
- Each worker:
    * instantiates its own langid + tiktoken encoder
    * filters chunk rows for English (in-worker)
    * tokenizes each kept row (per-row tokenization)
    * optionally appends EOS token after each row
    * saves tokens as .npy atomically (temp -> os.replace)
    * returns a small verification sample and (if PROFILE=True) timing counters
- Main process collects samples, writes a CSV profiling log (if enabled)
  and finally merges chunk .npy files into a memmapped merged .npy suitable
  for downstream training.

The merge step copies data in small blocks to avoid loading whole chunks into RAM.
"""

from __future__ import annotations

import os
import time
import uuid
import csv
import logging
import re
import gc
from pathlib import Path
from typing import List, Tuple, Optional, Iterator
from time import perf_counter

import numpy as np
from joblib import Parallel, delayed, parallel_backend

import pyarrow.dataset as ds
import pyarrow as pa

import langid
import tiktoken

# ----------------- Configuration -----------------
# Paths
INPUT_DIR = "datasets/100BT"                   # directory with parquet files
PARQUET_GLOB = "*.parquet"                   # file glob for parquet files
OUTPUT_DIR = Path("outputs/extract_v11")     # where chunk .npy files go
MERGED_OUTPUT = OUTPUT_DIR / "tokens_merged.npy"

# Chunking / streaming
BATCH_ROWS = 4096                             # rows per pyarrow RecordBatch
TARGET_CHUNK_BYTES = 50_000_000               # approximate bytes per dispatched chunk

# Parallelism / joblib
TOKENIZER_MODEL = "gpt2"
N_JOBS = max(1, min(os.cpu_count() // 2 or 1, 16)) # processes (defaults to number of logical CPUs)
PREFETCH_FACTOR = 4                            # how many chunks to prefetch (decrease if you run out of memory)
PARALLEL_VERBOSE = 5                           # joblib verbose (0 / 5 / 10)

# Language detection
ASCII_THRESHOLD = 0.5
LANG_MODE = "skip"

# EOS handling (important for autoregressive pretraining)
ADD_EOS = True  # If True, append tokenizer's EOS token id after each input row.

# Profiling toggle (opt-in)
PROFILE = True
PROFILE_CSV = OUTPUT_DIR / "tokenization_profile.csv"  # per-chunk profiling CSV

# Merge tuning: how many tokens are copied in one block during merging
# int32 tokens -> block_tokens * 4 bytes RAM roughly
MERGE_BLOCK_TOKENS = 1_000_000  # ~4 MB blocks by default

# Tunables for atomic saves
ATOMIC_SAVE_MAX_RETRIES = 8
ATOMIC_SAVE_RETRY_DELAY = 0.2                 # seconds

# Ensure output dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------- Logging -----------------
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
run_id = int(time.time())
log_file = LOG_DIR / f"extract_{run_id}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

# Add file handler for persistent logs
file_handler = logging.FileHandler(str(log_file), mode="w", encoding="utf-8")
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%H:%M:%S"))
logger.addHandler(file_handler)
logger.info("Starting run; writing logs to %s", log_file)

# ----------------- Helper utilities -----------------


def salvage_stale_temp_files(out_dir: Path) -> int:
    """
    Repair leftover temp artifacts from older buggy runs (np.save behavior).
    Handles both "chunk_*.npy.tmp.npy" and "chunk_*.npy.tmp" patterns.
    Returns number of renamed / repaired files.
    """
    repaired = 0
    # Pattern 1: chunk_XXXX.npy.tmp.npy -> chunk_XXXX.npy
    for p in out_dir.glob("chunk_*.npy.tmp.npy"):
        final = Path(str(p).replace(".npy.tmp.npy", ".npy"))
        if not final.exists():
            try:
                os.replace(str(p), str(final))
                repaired += 1
            except Exception as e:
                logger.debug("Could not salvage %s: %s", p, e)

    # Pattern 2: chunk_XXXX.npy.tmp -> chunk_XXXX.npy
    for p in out_dir.glob("chunk_*.npy.tmp"):
        final = p.with_suffix("")  # drops the final '.tmp' suffix
        if not final.exists():
            try:
                os.replace(str(p), str(final))
                repaired += 1
            except Exception as e:
                logger.debug("Could not salvage %s: %s", p, e)

    if repaired:
        logger.info("Salvaged %d stale temp files in %s", repaired, out_dir)
    return repaired


def list_parquet_files(input_dir: str, pattern: str = "*.parquet") -> List[str]:
    """Return sorted list of parquet file paths for input_dir/pattern."""
    p = Path(input_dir)
    if not p.exists():
        raise FileNotFoundError(f"Input dir not found: {p}")
    return sorted(str(x) for x in p.glob(pattern))


def choose_string_column(dataset: ds.dataset) -> str:
    """
    Pick the best string column name in the dataset schema.
    Prefers common names ('text','content','body') and falls back to
    the first string column found.
    """
    schema = dataset.schema
    str_columns = [f.name for f in schema if pa.types.is_string(f.type)]
    for candidate in ("text", "content", "body"):
        if candidate in str_columns:
            return candidate
    if not str_columns:
        raise RuntimeError("No string column in parquet schema(s)")
    return str_columns[0]


def is_english_text(text: str, mode: str = "accuracy", ascii_threshold: float = 0.5) -> bool:
    """
    Heuristic language detection used in this pipeline.

    Modes:
      - "skip": accept everything (fastest).
      - "accuracy": use langid.classify, fall back to ascii fraction.
      - "ascii"/"fast": only ascii fraction check.
    """
    if mode is None:
        mode = "accuracy"
    mode = mode.lower().strip()

    if mode == "skip":
        try:
            logger.debug("is_english_text: skip mode — accepting text without checks")
        except Exception:
            pass
        return True

    if not text:
        return False

    def ascii_fraction(s: str) -> float:
        total = len(s)
        if total == 0:
            return 0.0
        ascii_count = sum(1 for c in s if ord(c) < 128)
        return ascii_count / total

    if mode in ("ascii", "fast"):
        return ascii_fraction(text) >= ascii_threshold

    if mode == "accuracy":
        try:
            lang, _score = langid.classify(text)
        except Exception:
            lang = None
        if lang == "en":
            return True
        return ascii_fraction(text) >= ascii_threshold

    raise ValueError(f"is_english_text: unknown mode '{mode}'. Choose 'skip', 'accuracy' or 'ascii'.")


# ----------------- Streaming / chunking (main process) -----------------


def iter_raw_text_chunks_from_parquets(
    input_dir: str,
    pattern: str = "*.parquet",
    batch_rows: int = 4096,
    target_chunk_bytes: int = 50_000_000,
) -> Iterator[Tuple[List[str], int]]:
    """
    Stream parquet rows and yield chunks of raw rows (list[str], chunk_id).

    We do NOT filter here; filtering is done in-worker (parallelized).
    We pack rows until the approximate UTF-8 byte-size reaches target_chunk_bytes.
    """
    file_list = list_parquet_files(input_dir, pattern)
    if not file_list:
        logger.warning("No parquet files found for pattern %s in %s", pattern, input_dir)
        return

    dataset = ds.dataset(file_list, format="parquet")
    chosen_col = choose_string_column(dataset)
    logger.info("Using string column '%s' for text extraction", chosen_col)

    scanner = dataset.scanner(batch_size=batch_rows)
    chunk_id = 0
    buffer_texts: List[str] = []
    buffer_bytes = 0

    for record_batch in scanner.to_batches():
        arr = record_batch[chosen_col]
        for raw in arr.to_pylist():
            if raw is None:
                continue
            text = str(raw).strip()
            if not text:
                continue
            b = len(text.encode("utf-8"))

            if buffer_texts and (buffer_bytes + b) >= target_chunk_bytes:
                yield buffer_texts, chunk_id
                chunk_id += 1
                buffer_texts = []
                buffer_bytes = 0

            if b >= target_chunk_bytes:
                yield [text], chunk_id
                chunk_id += 1
                continue

            buffer_texts.append(text)
            buffer_bytes += b

    if buffer_texts:
        yield buffer_texts, chunk_id


# ----------------- Worker: filter (in-worker) + tokenize + save -----------------


def _safe_atomic_save_npy(arr: np.ndarray, out_path: Path,
                          max_retries: int = ATOMIC_SAVE_MAX_RETRIES,
                          retry_delay: float = ATOMIC_SAVE_RETRY_DELAY) -> None:
    """
    Safely write numpy array to out_path atomically (temp -> os.replace).

    Implementation notes:
    - Create a uniquely-named temporary file in the same directory.
    - Open a file handle and pass to np.save to avoid numpy appending ".npy".
    - Use os.replace to atomically publish the final file.
    - Retry a few times to tolerate transient Windows AV locks or network FS issues.
    """
    out_dir = out_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)

    tmp_name = f"{out_path.stem}.{os.getpid()}.{uuid.uuid4().hex}.npy.tmp"
    tmp_path = out_dir / tmp_name

    with open(tmp_path, "wb") as fh:
        np.save(fh, arr)

    if not tmp_path.exists():
        raise FileNotFoundError(f"Temp write failed: {tmp_path}")

    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            os.replace(str(tmp_path), str(out_path))
            return
        except (FileNotFoundError, PermissionError) as e:
            last_exc = e
            time.sleep(retry_delay)
        except Exception:
            raise
    raise last_exc or RuntimeError("Atomic replace failed for unknown reason")


def _get_eos_token_id(enc: "tiktoken.Encoding") -> Optional[int]:
    """
    Return EOS token ID for the given encoder if determinable; otherwise None.
    """
    try:
        try:
            ids = enc.encode("", allowed_special={"<|endoftext|>"})
            if ids:
                return int(ids[0])
        except Exception:
            pass
        try:
            ids = enc.encode("")
            if ids:
                return int(ids[0])
        except Exception:
            pass
    except Exception:
        pass

    try:
        if TOKENIZER_MODEL.lower().startswith("gpt2"):
            return 50256
    except Exception:
        pass

    return None


def tokenize_filter_and_save(
    chunk_texts: List[str],
    chunk_id: int,
    out_dir: str,
    tokenizer_model: str,
    sample_n: int = 20,
    lang_mode: str = LANG_MODE,
    ascii_threshold: float = ASCII_THRESHOLD,
    add_eos: bool = ADD_EOS,
) -> Optional[Tuple]:
    """
    Worker function executed inside a separate process.

    If PROFILE==True this returns an extended tuple:
      (out_path, token_count, sample_tokens, sample_text,
       lang_time, token_time, save_time, num_input_rows, num_kept_rows, chunk_id, total_time)

    If PROFILE==False it returns:
      (out_path, token_count, sample_tokens, sample_text)

    Returns None if no tokens were produced or on fatal error.
    """
    # Create tokenizer instance inside worker
    try:
        try:
            enc = tiktoken.get_encoding(tokenizer_model)
        except Exception:
            enc = tiktoken.encoding_for_model(tokenizer_model)
    except Exception as e:
        logger.exception("Worker failed to create tokenizer: %s", e)
        return None

    eos_id = _get_eos_token_id(enc) if add_eos else None
    if add_eos and eos_id is None:
        logger.warning("ADD_EOS requested but EOS token id couldn't be determined; EOS will be skipped.")

    # Timing counters (per-chunk)
    t_chunk_start = perf_counter()
    lang_time = 0.0
    token_time = 0.0
    save_time = 0.0

    # 1) Language detection per-row
    num_input_rows = 0
    kept_rows: List[str] = []
    for txt in chunk_texts:
        num_input_rows += 1
        if not txt:
            continue
        t0 = perf_counter()
        keep = is_english_text(txt, mode=lang_mode, ascii_threshold=ascii_threshold)
        lang_time += perf_counter() - t0
        if keep:
            kept_rows.append(txt)

    num_kept_rows = len(kept_rows)
    if num_kept_rows == 0:
        # nothing to save
        return None

    # 2) Tokenization per-row
    token_ids: List[int] = []
    for row in kept_rows:
        t0 = perf_counter()
        try:
            toks = enc.encode_ordinary(row)
        except Exception:
            toks = enc.encode(row)
        token_time += perf_counter() - t0
        if toks:
            token_ids.extend(toks)
            if eos_id is not None:
                token_ids.append(eos_id)

    if not token_ids:
        return None

    arr = np.array(token_ids, dtype=np.int32)
    out_path = Path(out_dir) / f"chunk_{chunk_id:06d}.npy"

    # 3) Atomic save (resumable)
    t_save_start = perf_counter()
    if out_path.exists():
        save_time = 0.0
        logger.debug("Chunk %s exists, skipping write", out_path.name)
    else:
        try:
            _safe_atomic_save_npy(arr, out_path)
            save_time = perf_counter() - t_save_start
        except Exception as e:
            logger.exception("Failed to save chunk %d -> %s : %s", chunk_id, out_path, e)
            # best-effort cleanup
            try:
                stem = out_path.stem
                patterns = [f"{stem}.*.npy.tmp", f"{stem}.*.npy.tmp.npy", f"{stem}.*.tmp"]
                for pat in patterns:
                    for f in out_path.parent.glob(pat):
                        try:
                            f.unlink(missing_ok=True)
                        except Exception:
                            pass
            except Exception:
                pass
            return None

    # 4) Prepare sample for verification
    sample_tokens = arr[:sample_n].tolist()
    try:
        sample_text = enc.decode(sample_tokens)
    except Exception:
        sample_text = " ".join(str(t) for t in sample_tokens)

    total_elapsed = perf_counter() - t_chunk_start

    if PROFILE:
        return (
            str(out_path),
            int(arr.size),
            sample_tokens,
            sample_text,
            float(lang_time),
            float(token_time),
            float(save_time),
            int(num_input_rows),
            int(num_kept_rows),
            int(chunk_id),
            float(total_elapsed),
        )
    else:
        return str(out_path), int(arr.size), sample_tokens, sample_text


# ----------------- Merge chunk files (main process) -----------------


def merge_chunks(chunk_files: List[str], out_path: Path, block_tokens: int = MERGE_BLOCK_TOKENS) -> None:
    """
    Merge numerically sorted chunk .npy files into one large memmapped int32 .npy
    without loading entire chunks into RAM.

    Parameters
    ----------
    chunk_files : List[str]
        Ordered list of .npy chunk file paths to merge.
    out_path : Path
        Destination .npy path for the merged memmap.
    block_tokens : int
        Number of token elements copied at once. Tune this to control peak RAM.
        (int32 tokens -> memory usage ≈ block_tokens * 4 bytes).
    """
    try:
        import winsound
        duration = 1000  # milliseconds
        freq = 1000  # Hz
        winsound.Beep(freq, duration)
    except:
        pass

    if not chunk_files:
        raise ValueError("No chunk files provided to merge")

    # Compute total length by opening each chunk as mmap (no full read)
    total = 0
    dtypes = []
    shapes = []
    for f in chunk_files:
        src = np.load(f, mmap_mode="r")
        shapes.append(src.shape[0])
        dtypes.append(src.dtype)
        total += src.shape[0]
        # close memmap reference
        del src

    # Choose dtype (ensure consistency)
    dtype = dtypes[0]
    if not all(dt == dtype for dt in dtypes):
        logger.warning("Chunk dtypes differ; casting to %s", dtype)

    # Create final memmap
    merged = np.memmap(str(out_path), dtype=dtype, mode="w+", shape=(total,))
    cursor = 0

    # Copy each chunk in small blocks
    for idx, f in enumerate(chunk_files):
        logger.info("Merging chunk %d/%d: %s", idx + 1, len(chunk_files), f)
        src = np.load(f, mmap_mode="r")
        n = src.shape[0]
        start = 0
        while start < n:
            end = start + block_tokens
            if end > n:
                end = n
            # read a slice from src (this creates a small temporary array)
            block = src[start:end]
            merged[cursor: cursor + (end - start)] = block
            cursor += (end - start)
            start = end
        del src  # close memmap-backed view to free file handle
        gc.collect()

    merged.flush()
    del merged
    gc.collect()
    logger.info("Merged %d chunks (%d tokens) -> %s", len(chunk_files), total, out_path)


# ----------------- Orchestration (main) -----------------


def run_pipeline():
    """
    Main orchestration:
      - Salvage stale temp files
      - Stream / chunk parquet rows
      - Dispatch chunks in batches to joblib workers
      - Collect profiling info (if enabled) and save to CSV
      - Merge chunk files into final memmapped array (streamed copy)
    """
    logger.info("Pipeline start: INPUT=%s OUTPUT=%s N_JOBS=%d", INPUT_DIR, OUTPUT_DIR, N_JOBS)

    # Repair leftovers to avoid accidental re-processing
    salvage_stale_temp_files(OUTPUT_DIR)

    chunk_gen = iter_raw_text_chunks_from_parquets(
        INPUT_DIR, PARQUET_GLOB, batch_rows=BATCH_ROWS, target_chunk_bytes=TARGET_CHUNK_BYTES
    )

    saved_files: List[str] = []
    total_tokens = 0

    # Profiling aggregates
    total_lang_time = 0.0
    total_token_time = 0.0
    total_save_time = 0.0
    total_chunks_profiled = 0
    total_input_rows = 0
    total_kept_rows = 0

    max_pending = max(1, N_JOBS * PREFETCH_FACTOR)

    # Prepare CSV if profiling
    csv_file = None
    csv_writer = None
    if PROFILE:
        write_header = not PROFILE_CSV.exists()
        csv_file = open(PROFILE_CSV, "a", newline="", encoding="utf-8")
        csv_writer = csv.writer(csv_file)
        if write_header:
            csv_writer.writerow([
                "chunk_id", "num_input_rows", "num_kept_rows", "token_count",
                "lang_time_s", "token_time_s", "save_time_s", "total_time_s"
            ])
            csv_file.flush()

    try:
        while True:
            # Prefetch up to max_pending chunks
            to_dispatch: List[Tuple[List[str], int]] = []
            for _ in range(max_pending):
                try:
                    chunk_texts, chunk_id = next(chunk_gen)
                    to_dispatch.append((chunk_texts, chunk_id))
                except StopIteration:
                    break

            if not to_dispatch:
                break

            logger.info("Dispatching %d chunks to workers (n_jobs=%d)", len(to_dispatch), N_JOBS)
            with parallel_backend("loky"):
                results = Parallel(
                    n_jobs=N_JOBS,
                    backend="loky",
                    prefer="processes",
                    verbose=PARALLEL_VERBOSE
                )(
                    delayed(tokenize_filter_and_save)(
                        chunk_texts, chunk_id, str(OUTPUT_DIR), TOKENIZER_MODEL, 20,
                        LANG_MODE, ASCII_THRESHOLD, ADD_EOS
                    )
                    for chunk_texts, chunk_id in to_dispatch
                )

            # Collect results
            for res in results:
                if res is None:
                    continue

                if PROFILE:
                    (
                        out_path, token_count, sample_tokens, sample_text,
                        lang_time, token_time, save_time,
                        num_input_rows, num_kept_rows, chunk_id, total_elapsed
                    ) = res

                    # write CSV row
                    csv_writer.writerow([
                        int(chunk_id),
                        int(num_input_rows),
                        int(num_kept_rows),
                        int(token_count),
                        f"{lang_time:.6f}",
                        f"{token_time:.6f}",
                        f"{save_time:.6f}",
                        f"{total_elapsed:.6f}",
                    ])
                    csv_file.flush()

                    # accumulate
                    total_lang_time += lang_time
                    total_token_time += token_time
                    total_save_time += save_time
                    total_chunks_profiled += 1
                    total_input_rows += num_input_rows
                    total_kept_rows += num_kept_rows
                    total_tokens += token_count

                else:
                    out_path, token_count, sample_tokens, sample_text = res
                    total_tokens += token_count

                saved_files.append(out_path)
                short = sample_text if len(sample_text) <= 200 else sample_text[:200] + "…[truncated]"
                logger.info("Chunk saved: %s — tokens=%d — sample tokens=%s", out_path, token_count, sample_tokens)
                logger.info("Chunk decoded sample -> %s", repr(short))

            gc.collect()

        # Fallback discovery in case we salvaged/created files but results list is empty
        if not saved_files:
            saved_files = [str(p) for p in OUTPUT_DIR.glob("chunk_*.npy")]

        if not saved_files:
            logger.info("No chunk files found/produced.")
            return

        # Profiling summary
        if PROFILE and total_chunks_profiled > 0:
            logger.info("=== Profiling summary ===")
            logger.info("Chunks profiled: %d", total_chunks_profiled)
            logger.info("Total input rows: %d, total kept rows: %d, total tokens: %d",
                        total_input_rows, total_kept_rows, total_tokens)
            logger.info("Total lang detection time: %.3fs", total_lang_time)
            logger.info("Total tokenization time: %.3fs", total_token_time)
            logger.info("Total save time: %.3fs", total_save_time)
            total_work = total_lang_time + total_token_time + total_save_time
            if total_work > 0:
                logger.info("Lang detection: %.2f%% ; Tokenization: %.2f%% ; Save: %.2f%%",
                            100.0 * total_lang_time / total_work,
                            100.0 * total_token_time / total_work,
                            100.0 * total_save_time / total_work)
            logger.info("Avg lang time / chunk: %.4fs", total_lang_time / total_chunks_profiled)
            logger.info("Avg token time / chunk: %.4fs", total_token_time / total_chunks_profiled)
            if total_token_time > 0:
                logger.info("Approx tokens/sec during tokenization: %.1f tok/s", total_tokens / total_token_time)

        # numeric sort by chunk id extracted from filename
        saved_files.sort(key=lambda f: int(re.sub(r"\D", "", Path(f).stem)))
        logger.info("Produced %d chunk files (approx total tokens=%d). Starting merge...", len(saved_files), total_tokens)

        # Use streamed merge with small blocks to avoid high RAM usage
        merge_chunks(saved_files, MERGED_OUTPUT, block_tokens=MERGE_BLOCK_TOKENS)
        logger.info("Final merged array at: %s", MERGED_OUTPUT)

    except Exception:
        logger.exception("Pipeline failed")
        raise
    finally:
        if csv_file:
            try:
                csv_file.close()
            except Exception:
                pass


if __name__ == "__main__":
    t0 = time.time()
    run_pipeline()
    logger.info("All done in %.1fs", time.time() - t0)

    try:
        import winsound
        duration = 100  # milliseconds
        freq = 200  # Hz
        while freq<10000:
            winsound.Beep(freq, duration)
            freq = int(freq * 1.1) + 1
    except:
        pass


03:45:12 [INFO] Starting run; writing logs to logs\extract_1756086312.log
03:45:12 [INFO] Pipeline start: INPUT=datasets/100BT OUTPUT=outputs\extract_v11 N_JOBS=8
03:45:12 [INFO] Using string column 'text' for text extraction
03:45:23 [INFO] Dispatching 32 chunks to workers (n_jobs=8)
[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:    7.4s
[Parallel(n_jobs=8)]: Done  24 out of  32 | elapsed:   20.9s remaining:    6.9s
[Parallel(n_jobs=8)]: Done  32 out of  32 | elapsed:   26.7s finished
03:45:49 [INFO] Chunk saved: outputs\extract_v11\chunk_000000.npy — tokens=10850108 — sample tokens=[464, 13362, 12091, 198, 1890, 477, 262, 1842, 11, 19661, 290, 10731, 287, 12091, 2517, 268, 447, 247, 82, 3835]
03:45:49 [INFO] Chunk decoded sample -> 'The Independent Jane\nFor all the love, romance and scandal in Jane Austen’s books'
03:45:49 [INFO] Chunk saved: outputs\extract_v11\chunk_000001.npy — tokens=10860493 — samp

In [2]:
try:
    import winsound
    duration = 100  # milliseconds
    freq = 200  # Hz
    while freq<10000:
        winsound.Beep(freq, duration)
        freq = int(freq * 1.1) + 1
except:
    pass
